# Trenowanie modeli

## Przygotowanie danych do treningu

In [3]:
import os, json, random, shutil, cv2
from pathlib import Path
import yaml

# Kategorie (foldery) – pobierz z nagrań lub metadata
categories = [
    "city_daylight",
    "city_daylight_rain",
    "city_night",
    "city_night_rain",
    "highway_daylight",
    "highway_daylight_rain",
    "highway_night",
    "highway_night_rain",
    "mountain_roads_daylight",
    "mountain_roads_daylight_rain",
    "mountain_roads_night",
    "mountain_roads_night_rain",
    "roadworks_daylight",
    "roadworks_daylight_rain",
    "roadworks_night",
    "roadworks_night_rain",
]

# Ścieżki
VIDEO_BASE = Path(r"D:\MGR\Master\recordings")
META_BASE = Path("../data/metadata")
DATASET_ALL = Path("../data/dataset")
IMAGES_ALL = DATASET_ALL / "images" / "all"
LABELS_ALL = DATASET_ALL / "labels" / "all"

# Tworzymy katalogi
for p in [IMAGES_ALL, LABELS_ALL]:
    p.mkdir(parents=True, exist_ok=True)
print("Ścieżki utworzone.")

Ścieżki utworzone.


In [ ]:
# Ekstrakcja pełnych klatek + adnotacji


def build_fullframe_dataset(video_base, meta_base, images_out, labels_out):
    for cat in categories:
        mp4_dir = video_base / cat / "mp4"
        md_dir = meta_base / cat
        if not mp4_dir.is_dir() or not md_dir.is_dir():
            print(f"[WARN] Brak folderu mp4/ lub metadata/ dla {cat}, pomijam.")
            continue

        # Przejdź po wszystkich plikach json w md_dir
        for json_file in md_dir.glob("*.json"):
            base_name = json_file.stem  # np. "20240308_143701F_c_d"
            # odpowiadający plik mp4
            mp4_name = base_name.replace("_", " ").upper() + ".mp4"
            video_path = mp4_dir / mp4_name
            if not video_path.exists():
                print(f"[WARN] Brak wideo {mp4_name} w {mp4_dir}")
                continue

            # Wczytujemy metadane
            with open(json_file, "r", encoding="utf-8") as f:
                metadata = json.load(f)["metadata"]

            cap = cv2.VideoCapture(str(video_path))
            fps = cap.get(cv2.CAP_PROP_FPS)
            ih, iw = None, None

            # Dla każdej dziury
            for key, entry in metadata.items():
                t = entry["z"][0]
                frame_no = int(t * fps)
                cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
                ret, frame = cap.read()
                if not ret:
                    continue
                ih, iw = frame.shape[:2]

                # bbox z metadanych
                x, y, w, h = entry["xy"][1:]
                x1, y1 = x, y
                x2, y2 = x + w, y + h

                # nazwa pliku wynikowego
                fname = f"{cat}_{frame_no}_{key}.jpg"
                img_out = images_out / fname
                lbl_out = labels_out / fname.replace(".jpg", ".txt")

                # zapis ramki
                cv2.imwrite(str(img_out), frame)
                # zapis etykiety YOLO
                xc = (x1 + x2) / 2 / iw
                yc = (y1 + y2) / 2 / ih
                wr = (x2 - x1) / iw
                hr = (y2 - y1) / ih
                with open(lbl_out, "w") as l:
                    l.write(f"0 {xc:.6f} {yc:.6f} {wr:.6f} {hr:.6f}\n")

            cap.release()


# Uruchamiamy
build_fullframe_dataset(VIDEO_BASE, META_BASE, IMAGES_ALL, LABELS_ALL)
print("Ekstrakcja zakończona (poprawione ścieżki).")

Ekstrakcja zakończona (poprawione ścieżki).
